In [4]:

# ============================================================
# PHÂN LOẠI TIN NHẮN LỪA ĐẢO TRỰC TUYẾN
# TF-IDF + MULTINOMIAL NAIVE BAYES
# ============================================================

# 1. IMPORT THƯ VIỆN

import os
import json
import joblib
import pandas as pd

from datetime import datetime
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score


# 2. KHAI BÁO CẤU HÌNH

TRAIN_PATH = "/content/train.csv"
TEST_PATH = "/content/test.csv"

TEXT_COLUMN = "text_clean"
LABEL_COLUMN = "label"

MODEL_PATH = "/content/naive_bayes_pipeline.pkl"
PREDICTION_PATH = "/content/y_test_y_pred.csv"
INFO_PATH = "/content/model_info.json"

EXPECTED_LABELS = {"ham", "spam", "smishing"}


# 3. ĐỌC DỮ LIỆU

if not os.path.exists(TRAIN_PATH):
    raise FileNotFoundError("Không tìm thấy file train.csv")

if not os.path.exists(TEST_PATH):
    raise FileNotFoundError("Không tìm thấy file test.csv")

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Kích thước tập Train:", train_df.shape)
print("Kích thước tập Test:", test_df.shape)


# 4. KIỂM TRA DỮ LIỆU

required_columns = [TEXT_COLUMN, LABEL_COLUMN]

for column in required_columns:
    if column not in train_df.columns or column not in test_df.columns:
        raise ValueError(f"Thiếu cột dữ liệu: {column}")

if train_df[required_columns].isnull().any().any():
    raise ValueError("Tập Train có dữ liệu bị thiếu.")

if test_df[required_columns].isnull().any().any():
    raise ValueError("Tập Test có dữ liệu bị thiếu.")

train_df[TEXT_COLUMN] = train_df[TEXT_COLUMN].astype(str)
test_df[TEXT_COLUMN] = test_df[TEXT_COLUMN].astype(str)

if train_df[TEXT_COLUMN].str.strip().eq("").any():
    raise ValueError("Tập Train có nội dung tin nhắn rỗng.")

if test_df[TEXT_COLUMN].str.strip().eq("").any():
    raise ValueError("Tập Test có nội dung tin nhắn rỗng.")

if set(train_df[LABEL_COLUMN].unique()) != EXPECTED_LABELS:
    raise ValueError("Tập Train không có đủ ba nhãn hợp lệ.")

if set(test_df[LABEL_COLUMN].unique()) != EXPECTED_LABELS:
    raise ValueError("Tập Test không có đủ ba nhãn hợp lệ.")

print("\nPhân bố nhãn Train:")
print(train_df[LABEL_COLUMN].value_counts())

print("\nPhân bố nhãn Test:")
print(test_df[LABEL_COLUMN].value_counts())


# 5. XÁC ĐỊNH ĐẶC TRƯNG VÀ NHÃN

X_train = train_df[TEXT_COLUMN]
y_train = train_df[LABEL_COLUMN]

X_test = test_df[TEXT_COLUMN]
y_test = test_df[LABEL_COLUMN]

print("\nSố mẫu Train:", len(X_train))
print("Số mẫu Test:", len(X_test))


# 6. XÂY DỰNG PIPELINE TF-IDF + NAIVE BAYES

model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2,
            max_features=50000,
            sublinear_tf=True
        )
    ),
    (
        "naive_bayes",
        MultinomialNB(alpha=1.0)
    )
])


# 7. HUẤN LUYỆN MÔ HÌNH

model.fit(X_train, y_train)

print("\nHuấn luyện mô hình hoàn tất!")


# 8. DỰ ĐOÁN TRÊN TẬP TEST

y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)

print("Dự đoán hoàn tất!")


# 9. ĐÁNH GIÁ ĐỘ CHÍNH XÁC

accuracy = accuracy_score(y_test, y_pred)

print("\nAccuracy:", round(accuracy, 4))
print(f"Accuracy: {accuracy * 100:.2f}%")


# 10. LƯU KẾT QUẢ DỰ ĐOÁN

prediction_df = pd.DataFrame({
    "y_test": y_test.reset_index(drop=True),
    "y_pred": y_pred
})

if "id" in test_df.columns:
    prediction_df.insert(
        0,
        "id",
        test_df["id"].reset_index(drop=True)
    )

prediction_df["text_clean"] = X_test.reset_index(drop=True)

class_labels = model.named_steps["naive_bayes"].classes_

for index, label in enumerate(class_labels):
    prediction_df[f"probability_{label}"] = y_pred_proba[:, index]

prediction_df.to_csv(
    PREDICTION_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\nĐã lưu kết quả dự đoán:", PREDICTION_PATH)


# 11. LƯU PIPELINE MÔ HÌNH

joblib.dump(model, MODEL_PATH)

print("Đã lưu mô hình:", MODEL_PATH)


# 12. KIỂM TRA MÔ HÌNH ĐÃ LƯU

loaded_model = joblib.load(MODEL_PATH)

sample_message = X_test.iloc[0]

sample_prediction = loaded_model.predict([sample_message])[0]

print("\nTin nhắn kiểm tra:")
print(sample_message)

print("Nhãn dự đoán:", sample_prediction)


# 13. LƯU THÔNG TIN MÔ HÌNH

model_info = {
    "project": "Online Scam Message Classification",
    "algorithm": "Multinomial Naive Bayes",
    "vectorizer": "TF-IDF",
    "text_column": TEXT_COLUMN,
    "label_column": LABEL_COLUMN,
    "train_samples": len(X_train),
    "test_samples": len(X_test),
    "classes": [str(label) for label in class_labels],
    "accuracy": float(accuracy),
    "tfidf_parameters": {
        "lowercase": True,
        "ngram_range": [1, 2],
        "min_df": 2,
        "max_features": 50000,
        "sublinear_tf": True
    },
    "naive_bayes_parameters": {
        "alpha": 1.0
    },
    "created_at": datetime.now().isoformat()
}

with open(INFO_PATH, "w", encoding="utf-8") as file:
    json.dump(model_info, file, ensure_ascii=False, indent=4)


# 14. TỔNG KẾT

print("\n" + "=" * 50)
print("KẾT QUẢ HUẤN LUYỆN MÔ HÌNH")
print("=" * 50)

print("Thuật toán: Multinomial Naive Bayes")
print("Vectorizer: TF-IDF")
print("Số mẫu Train:", len(X_train))
print("Số mẫu Test:", len(X_test))
print(f"Accuracy: {accuracy * 100:.2f}%")

print("\nCác file kết quả:")

for file_path in [MODEL_PATH, PREDICTION_PATH, INFO_PATH]:
    print(
        f"{os.path.basename(file_path)}: "
        f"{'Đã tạo' if os.path.exists(file_path) else 'Chưa tạo'}"
    )

print("\nHoàn tất!")


Kích thước tập Train: (4637, 8)
Kích thước tập Test: (1160, 8)

Phân bố nhãn Train:
label
ham         3867
smishing     422
spam         348
Name: count, dtype: int64

Phân bố nhãn Test:
label
ham         967
smishing    106
spam         87
Name: count, dtype: int64

Số mẫu Train: 4637
Số mẫu Test: 1160

Huấn luyện mô hình hoàn tất!
Dự đoán hoàn tất!

Accuracy: 0.925
Accuracy: 92.50%

Đã lưu kết quả dự đoán: /content/y_test_y_pred.csv
Đã lưu mô hình: /content/naive_bayes_pipeline.pkl

Tin nhắn kiểm tra:
i am in escape theatre now. . going to watch kavalan in a few minutes
Nhãn dự đoán: ham

KẾT QUẢ HUẤN LUYỆN MÔ HÌNH
Thuật toán: Multinomial Naive Bayes
Vectorizer: TF-IDF
Số mẫu Train: 4637
Số mẫu Test: 1160
Accuracy: 92.50%

Các file kết quả:
naive_bayes_pipeline.pkl: Đã tạo
y_test_y_pred.csv: Đã tạo
model_info.json: Đã tạo

Hoàn tất!


In [5]:

import pandas as pd

# Đọc dữ liệu
test_df = pd.read_csv("/content/test.csv")
result_df = pd.read_csv("/content/y_test_y_pred.csv")

# Lấy các mẫu dự đoán sai
wrong_df = result_df[result_df["y_test"] != result_df["y_pred"]].copy()

print("Tổng số mẫu dự đoán sai:", len(wrong_df))

# Thống kê các trường hợp nhầm lẫn
print("\nThống kê nhãn thực tế và nhãn dự đoán:")
print(
    pd.crosstab(
        wrong_df["y_test"],
        wrong_df["y_pred"],
        rownames=["Nhãn thực tế"],
        colnames=["Nhãn dự đoán"]
    )
)

# Hiển thị 20 mẫu dự đoán sai
display(
    wrong_df[
        ["text_clean", "y_test", "y_pred"]
    ].head(20)
)

# Xuất file để phân tích
wrong_df.to_csv(
    "/content/misclassified_samples.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nĐã lưu misclassified_samples.csv")


Tổng số mẫu dự đoán sai: 87

Thống kê nhãn thực tế và nhãn dự đoán:
Nhãn dự đoán  ham  smishing  spam
Nhãn thực tế                     
smishing       28         0     2
spam           52         5     0


,text_clean,y_test,y_pred
2,welcome to uk-mobile-date this msg is free giv...,spam,ham
12,hi ya babe x u 4goten bout me?' scammers getti...,spam,ham
25,"congratulations, your cv was reviewed and you ...",smishing,ham
33,2/2 146tf150p win tickets..call 08701417012,spam,ham
43,87077: kick off a new season with 2wks free go...,spam,ham
46,how about getting in touch with folks waiting ...,spam,ham
56,in the simpsons movie released in july 2007 na...,spam,ham
58,sunshine hols. to claim ur med holiday send a ...,spam,ham
69,"did you hear about the new ""divorce barbie""? i...",spam,ham
85,hey boys. want hot xxx pics sent direct 2 ur p...,spam,ham



Đã lưu misclassified_samples.csv


In [6]:

# ============================================================
# THỬ NGHIỆM LẦN 2: CẢI THIỆN MÔ HÌNH NAIVE BAYES
# TF-IDF + RANDOM OVERSAMPLING + MULTINOMIAL NAIVE BAYES
# ============================================================

import pandas as pd
import joblib

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import RandomOverSampler

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Đọc lại dữ liệu gốc
train_df = pd.read_csv("/content/train.csv")
test_df = pd.read_csv("/content/test.csv")

X_train = train_df["text_clean"].astype(str)
y_train = train_df["label"]

X_test = test_df["text_clean"].astype(str)
y_test = test_df["label"]

# Xây dựng Pipeline
model_v2 = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2,
            max_features=50000,
            sublinear_tf=True
        )
    ),
    (
        "oversampling",
        RandomOverSampler(random_state=42)
    ),
    (
        "naive_bayes",
        MultinomialNB(alpha=1.0)
    )
])

# Huấn luyện
model_v2.fit(X_train, y_train)

# Dự đoán
y_pred_v2 = model_v2.predict(X_test)

# Đánh giá
accuracy_v2 = accuracy_score(y_test, y_pred_v2)

print("ACCURACY LẦN 2:", round(accuracy_v2 * 100, 2), "%")

print("\nCLASSIFICATION REPORT:")
print(
    classification_report(
        y_test,
        y_pred_v2,
        zero_division=0
    )
)

print("\nCONFUSION MATRIX:")
print(
    confusion_matrix(
        y_test,
        y_pred_v2,
        labels=["ham", "spam", "smishing"]
    )
)

# Lưu mô hình lần 2
joblib.dump(
    model_v2,
    "/content/naive_bayes_pipeline_v2.pkl"
)

# Lưu kết quả dự đoán lần 2
result_v2 = pd.DataFrame({
    "y_test": y_test.reset_index(drop=True),
    "y_pred": y_pred_v2
})

result_v2.to_csv(
    "/content/y_test_y_pred_v2.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nĐã lưu mô hình và kết quả lần 2.")


ACCURACY LẦN 2: 96.21 %

CLASSIFICATION REPORT:
              precision    recall  f1-score   support

         ham       0.99      0.98      0.99       967
    smishing       0.90      0.90      0.90       106
        spam       0.78      0.79      0.78        87

    accuracy                           0.96      1160
   macro avg       0.89      0.89      0.89      1160
weighted avg       0.96      0.96      0.96      1160


CONFUSION MATRIX:
[[952  10   5]
 [ 12  69   6]
 [  1  10  95]]

Đã lưu mô hình và kết quả lần 2.


In [7]:

import joblib
import pandas as pd

# Tải mô hình V2
model_v2 = joblib.load(
    "/content/naive_bayes_pipeline_v2.pkl"
)

# Một số tin nhắn thử nghiệm
test_messages = [
    "Congratulations! You have won a free prize. Call now!",
    "Hey, are you coming to class tomorrow?",
    "Your bank account has been suspended. Click this link immediately."
]

# Dự đoán
predictions = model_v2.predict(test_messages)
probabilities = model_v2.predict_proba(test_messages)

# Hiển thị kết quả
for message, prediction, probability in zip(
    test_messages,
    predictions,
    probabilities
):
    print("-" * 60)
    print("Tin nhắn:", message)
    print("Dự đoán:", prediction)
    print("Xác suất:")

    for label, prob in zip(model_v2.classes_, probability):
        print(f"  {label}: {prob:.4f}")


------------------------------------------------------------
Tin nhắn: Congratulations! You have won a free prize. Call now!
Dự đoán: smishing
Xác suất:
  ham: 0.0004
  smishing: 0.9959
  spam: 0.0036
------------------------------------------------------------
Tin nhắn: Hey, are you coming to class tomorrow?
Dự đoán: ham
Xác suất:
  ham: 0.9817
  smishing: 0.0111
  spam: 0.0073
------------------------------------------------------------
Tin nhắn: Your bank account has been suspended. Click this link immediately.
Dự đoán: smishing
Xác suất:
  ham: 0.0021
  smishing: 0.9959
  spam: 0.0020


In [8]:

# ============================================================
# ĐÁNH GIÁ MÔ HÌNH NAIVE BAYES V2
# ============================================================

import pandas as pd
import json

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

# 1. Đọc kết quả dự đoán lần 2
result = pd.read_csv("/content/y_test_y_pred_v2.csv")

y_test = result["y_test"]
y_pred = result["y_pred"]

# 2. Khai báo thứ tự nhãn
labels = ["ham", "spam", "smishing"]

# 3. Tính Accuracy
accuracy = accuracy_score(y_test, y_pred)

# 4. Tính các chỉ số đánh giá
precision, recall, f1, _ = precision_recall_fscore_support(
    y_test,
    y_pred,
    labels=labels,
    average="macro",
    zero_division=0
)

print("=" * 55)
print("KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH V2")
print("=" * 55)

print(f"Accuracy       : {accuracy:.4f}")
print(f"Macro Precision: {precision:.4f}")
print(f"Macro Recall   : {recall:.4f}")
print(f"Macro F1-score : {f1:.4f}")

# 5. Classification Report
print("\nCLASSIFICATION REPORT:")
report = classification_report(
    y_test,
    y_pred,
    labels=labels,
    zero_division=0
)

print(report)

# 6. Confusion Matrix
print("CONFUSION MATRIX:")
cm = confusion_matrix(
    y_test,
    y_pred,
    labels=labels
)

cm_df = pd.DataFrame(
    cm,
    index=labels,
    columns=labels
)

print(cm_df)

# 7. Lưu kết quả đánh giá
evaluation = {
    "model": "TF-IDF + RandomOverSampler + MultinomialNB",
    "accuracy": float(accuracy),
    "macro_precision": float(precision),
    "macro_recall": float(recall),
    "macro_f1": float(f1),
    "confusion_matrix": cm.tolist(),
    "labels": labels
}

with open(
    "/content/evaluation_v2.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(evaluation, f, ensure_ascii=False, indent=4)

cm_df.to_csv(
    "/content/confusion_matrix_v2.csv",
    encoding="utf-8-sig"
)

print("\nĐã lưu evaluation_v2.json")
print("Đã lưu confusion_matrix_v2.csv")


KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH V2
Accuracy       : 0.9621
Macro Precision: 0.8860
Macro Recall   : 0.8913
Macro F1-score : 0.8886

CLASSIFICATION REPORT:
              precision    recall  f1-score   support

         ham       0.99      0.98      0.99       967
        spam       0.78      0.79      0.78        87
    smishing       0.90      0.90      0.90       106

    accuracy                           0.96      1160
   macro avg       0.89      0.89      0.89      1160
weighted avg       0.96      0.96      0.96      1160

CONFUSION MATRIX:
          ham  spam  smishing
ham       952    10         5
spam       12    69         6
smishing    1    10        95

Đã lưu evaluation_v2.json
Đã lưu confusion_matrix_v2.csv
